In [1]:
!apt-get remove google-chrome-stable -y
!rm -rf /usr/bin/google-chrome
!rm -rf /usr/bin/chromedriver


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following packages were automatically installed and are no longer required:
  libvulkan1 mesa-vulkan-drivers
Use 'apt autoremove' to remove them.
The following packages will be REMOVED:
  google-chrome-stable
0 upgraded, 0 newly installed, 1 to remove and 35 not upgraded.
After this operation, 375 MB disk space will be freed.
(Reading database ... 125199 files and directories currently installed.)
Removing google-chrome-stable (134.0.6998.117-1) ...
Processing triggers for mailcap (3.70+nmu1ubuntu1) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
!apt-get update
!apt-get install -y wget unzip
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb
!apt-get -f install -y
!google-chrome --version  # Verify Chrome installation


Get:1 https://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,215 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,762 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2,692 kB]
Hit:14

In [3]:
!wget https://storage.googleapis.com/chrome-for-testing-public/134.0.6998.117/linux64/chromedriver-linux64.zip
!unzip chromedriver-linux64.zip
!mv chromedriver-linux64/chromedriver /usr/bin/chromedriver
!chmod +x /usr/bin/chromedriver
!chromedriver --version  # Verify ChromeDriver installation


--2025-03-20 19:35:27--  https://storage.googleapis.com/chrome-for-testing-public/134.0.6998.117/linux64/chromedriver-linux64.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 142.251.188.207, 192.178.163.207, 74.125.142.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.251.188.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9521768 (9.1M) [application/zip]
Saving to: ‘chromedriver-linux64.zip.2’

chromedriver-linux6 100%[===================>]   9.08M  --.-KB/s    in 0.05s   

2025-03-20 19:35:27 (179 MB/s) - ‘chromedriver-linux64.zip.2’ saved [9521768/9521768]

Archive:  chromedriver-linux64.zip
replace chromedriver-linux64/LICENSE.chromedriver? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: chromedriver-linux64/LICENSE.chromedriver  
replace chromedriver-linux64/THIRD_PARTY_NOTICES.chromedriver? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: chromedriver-linux64/THIRD_PARTY_NOTICES.chromedriver  
  infl

In [4]:
!pip install selenium

In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

# Configure Chrome options
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.binary_location = "/usr/bin/google-chrome"  # Correct Chrome binary

# Initialize WebDriver with the matching ChromeDriver
service = Service("/usr/bin/chromedriver")
driver = webdriver.Chrome(service=service, options=chrome_options)

# Test WebDriver
driver.get("https://www.google.com")
print("✅ ChromeDriver is working correctly!")
driver.quit()


✅ ChromeDriver is working correctly!


In [6]:
import os
import requests
import pandas as pd
import time
import random
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

# User-Agent List to avoid bot detection
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/100.0.4896.127 Safari/537.36",
]

def get_headers():
    """Returns a random User-Agent header for requests."""
    return {"User-Agent": random.choice(USER_AGENTS)}

# Set up Selenium WebDriver
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run in headless mode
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.binary_location = "/usr/bin/google-chrome"

service = Service("/usr/bin/chromedriver")
driver = webdriver.Chrome(service=service, options=chrome_options)

# Base URLs for PolitiFact's different fact-check ratings
RULINGS = {
    "True": "https://www.politifact.com/factchecks/list/?ruling=true&page=",
    "Mostly True": "https://www.politifact.com/factchecks/list/?ruling=mostly-true&page=",
    "Half True": "https://www.politifact.com/factchecks/list/?ruling=half-true&page=",
    "Mostly False": "https://www.politifact.com/factchecks/list/?ruling=barely-true&page=",
    "False": "https://www.politifact.com/factchecks/list/?ruling=false&page=",
    "Pants on Fire": "https://www.politifact.com/factchecks/list/?ruling=pants-fire&page=",
}

NUM_PAGES = 20  # Number of pages to scrape
all_data = []  # Store all extracted data

# Step 1: Extract Articles from Fact-Check Lists
for ruling, base_url in RULINGS.items():
    for page in range(1, NUM_PAGES + 1):
        url = base_url + str(page)
        try:
            driver.get(url)
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(5)

            WebDriverWait(driver, 40).until(
                EC.presence_of_element_located((By.XPATH, "//article[contains(@class, 'm-statement')]"))
            )
        except Exception as e:
            print(f"Retrying page {page} for ruling '{ruling}' due to timeout...")
            time.sleep(10)
            continue

        articles = driver.find_elements(By.XPATH, "//article[contains(@class, 'm-statement')]")
        for article in articles:
            try:
                # Extract the claim (statement)
                claim_element = article.find_element(By.CLASS_NAME, "m-statement__quote")
                claim_text = claim_element.text.strip()

                # Extract the article URL correctly
                try:
                    quote_div = article.find_element(By.CLASS_NAME, "m-statement__quote")
                    claim_a_tag = quote_div.find_element(By.TAG_NAME, "a")  # Get <a> inside <div class="m-statement__quote">
                    claim_url = claim_a_tag.get_attribute("href")

                    # Ensure the URL is absolute
                    if claim_url.startswith("/"):
                        claim_url = "https://www.politifact.com" + claim_url
                except:
                    claim_url = "N/A"

                all_data.append({"Article URL": claim_url, "Claim": claim_text, "Rating": ruling})
            except Exception as e:
                print(f"Skipping article due to error: {e}")

# Close Selenium WebDriver
driver.quit()

# Convert to DataFrame and save intermediate CSV
df = pd.DataFrame(all_data)
df.to_csv("politifact_articles.csv", index=False)

# Step 2: Extract Image URLs from Each Article
def extract_image_url(article_url):
    try:
        for _ in range(3):  # Retry up to 3 times in case of failure
            response = requests.get(article_url, headers=get_headers(), timeout=10)
            if response.status_code == 200:
                break  # Exit loop if request is successful
            time.sleep(5)  # Wait before retrying

        if response.status_code != 200:
            print(f"Skipping {article_url} due to repeated errors")
            return "N/A"

        soup = BeautifulSoup(response.text, "html.parser")

        # Step 1: Locate the main section <div class="t-row__center">
        main_section = soup.find("div", class_="t-row__center")
        if not main_section:
            print(f"No 't-row__center' found in {article_url}")
            return "N/A"

        # Step 2: Inside main section, find <article class="m-display__inner">
        article = main_section.find("article", class_="m-display__inner")
        if not article:
            print(f"No 'm-display__inner' found in {article_url}")
            return "N/A"

        # Step 3: Find the <picture> tag inside the article
        picture_tag = article.find("picture")
        if picture_tag:
            # Step 4: Extract the best quality image from <source>
            source_tag = picture_tag.find("source", {"media": "(min-width: 520px)"})
            if source_tag and "srcset" in source_tag.attrs:
                return source_tag["srcset"]

            # Step 5: Fallback to any available <source> if high-res is missing
            source_tag = picture_tag.find("source")
            if source_tag and "srcset" in source_tag.attrs:
                return source_tag["srcset"]

            # Step 6: Fallback to <img> tag inside <picture>
            img_tag = picture_tag.find("img")
            if img_tag and "src" in img_tag.attrs:
                return img_tag["src"]

        print(f"No image found for {article_url}")
        return "N/A"

    except Exception as e:
        print(f"Error fetching image URL from {article_url}: {e}")
        return "N/A"  # Return "N/A" if no valid image is found

# Step 3: Add Image URLs to DataFrame
df["Image URL"] = df["Article URL"].apply(extract_image_url)

# Save final dataset with images
df.to_csv("politifact_articles_with_images.csv", index=False)

print(f"✅ Scraping complete! Data saved in 'politifact_articles_with_images.csv'.")


No 'm-display__inner' found in https://www.politifact.com/factchecks/2025/mar/03/cavalier-johnson/mps-does-indeed-have-a-larger-tax-levy-than-the-ci/
No 'm-display__inner' found in https://www.politifact.com/factchecks/2024/sep/30/kamala-harris/trump-never-managed-to-implement-rx-drug-price-neg/
No 'm-display__inner' found in https://www.politifact.com/factchecks/2023/oct/18/instagram-posts/good-enough-to-be-true-lego-donated-model-mri-scan/
No 'm-display__inner' found in https://www.politifact.com/factchecks/2023/jun/07/instagram-posts/good-enough-to-be-true-young-boy-saved-two-lives-i/
No 'm-display__inner' found in https://www.politifact.com/factchecks/2023/mar/09/fentrice-driskell/1-every-3-ron-desantis-spends-federal-government-y/
No 'm-display__inner' found in https://www.politifact.com/factchecks/2022/nov/09/everytown-gun-safety/yes-arizona-republican-governor-candidate-kari-lak/
No 'm-display__inner' found in https://www.politifact.com/factchecks/2022/nov/04/ashley-hinson/did-n

In [9]:
df

,Article URL,Claim,Rating,Image URL
0,https://www.politifact.com/personalities/caval...,"“After the referendum, Milwaukee Public School...",True,N/A
1,https://www.politifact.com/personalities/joel-...,“90% of the (school) districts in Wisconsin al...,True,N/A
2,https://www.politifact.com/personalities/susan...,Wisconsin does not “require judges to automati...,True,N/A
3,https://www.politifact.com/personalities/commo...,Wisconsin makes it more difficult for its citi...,True,N/A
4,https://www.politifact.com/personalities/robin...,"Voter ID ""is supported, if you look at any pol...",True,N/A
...,...,...,...,...
295,https://www.politifact.com/personalities/faceb...,Videos show President-elect Donald Trump promi...,Pants on Fire,N/A
296,https://www.politifact.com/personalities/insta...,A “ballot mule” was caught on camera Nov. 2 in...,Pants on Fire,N/A
297,https://www.politifact.com/personalities/faceb...,"Gwinnett County, Georgia, sheriff's office sai...",Pants on Fire,N/A
298,https://www.politifact.com/personalities/tweets/,"""Haitianos afirman haber llegado a EEUU hace s...",Pants on Fire,N/A


In [10]:
df

,Article URL,Claim,Rating,Image URL
0,https://www.politifact.com/personalities/caval...,"“After the referendum, Milwaukee Public School...",True,N/A
1,https://www.politifact.com/personalities/joel-...,“90% of the (school) districts in Wisconsin al...,True,N/A
2,https://www.politifact.com/personalities/susan...,Wisconsin does not “require judges to automati...,True,N/A
3,https://www.politifact.com/personalities/commo...,Wisconsin makes it more difficult for its citi...,True,N/A
4,https://www.politifact.com/personalities/robin...,"Voter ID ""is supported, if you look at any pol...",True,N/A
...,...,...,...,...
295,https://www.politifact.com/personalities/faceb...,Videos show President-elect Donald Trump promi...,Pants on Fire,N/A
296,https://www.politifact.com/personalities/insta...,A “ballot mule” was caught on camera Nov. 2 in...,Pants on Fire,N/A
297,https://www.politifact.com/personalities/faceb...,"Gwinnett County, Georgia, sheriff's office sai...",Pants on Fire,N/A
298,https://www.politifact.com/personalities/tweets/,"""Haitianos afirman haber llegado a EEUU hace s...",Pants on Fire,N/A
